|                |   |
:----------------|---|
| **Nombres**     | Jocelyn Jiménez
| **Fecha**      | 04/05/26  |

# **Bootstrapping**

Bootstramping, muestras diferentes.
Se eligen filas aleatoriamente. Esto porque queremos simular múltiples muestras de la población.

-> Esto con el objetivo de estimar intervalos de confianza de mis parámetros.

  -> por significancia estadística

**1. Dataset**

In [2]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

df = pd.read_excel("Motor Trend Car Road Tests.xlsx")
df.head()

Saving Motor Trend Car Road Tests.xlsx to Motor Trend Car Road Tests.xlsx


,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


**2. Regresión lineal**

In [3]:
import statsmodels.api as sm

In [4]:
X = df[['hp', 'qsec']]
y = df['mpg']

In [5]:
X = sm.add_constant(X)

In [6]:
modelo = sm.OLS(y, X).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        22:40:47   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

- **2a. Intervalo de confianza**

In [7]:
intervalo_c = modelo.conf_int()
intervalo_c.columns = ['limite_inferior', 'limite_superior']
print(intervalo_c)

       limite_inferior  limite_superior
const        25.614894        71.032516
hp           -0.113089        -0.056097
qsec         -1.979929         0.206770


**3. Bootstrap**

*3a. Regresión en cada muestra*

In [8]:
import numpy as np

n = len(df)
betas = []

In [9]:
for i in range(1000):

    muestra_rem = df.sample(n=n, replace=True)

    X_b = muestra_rem[['hp', 'qsec']]
    y_b = muestra_rem['mpg']
    X_b = sm.add_constant(X_b)

    modelo_2 = sm.OLS(y_b, X_b).fit()

    betas.append(modelo_2.params.values)

In [10]:
betas = np.array(betas)

3b. Intervalos bootstrap

In [11]:
medias = betas.mean(axis=0)
std = betas.std(axis=0)

print("Medias:", medias)
print("Desviaciones:", std)

Medias: [50.36952077 -0.08792182 -0.98011597]
Desviaciones: [11.08655569  0.01703684  0.5289866 ]


In [12]:
Limite_I_boot = medias - 2 * std
Limite_S_boot = medias + 2 * std

In [13]:
intervalo_boot_df = pd.DataFrame({
    'limite_inferior_boot': Limite_I_boot,
    'limite_superior_boot': Limite_S_boot
}, index=['const', 'hp', 'qsec'])

print(intervalo_boot_df)

       limite_inferior_boot  limite_superior_boot
const             28.196409             72.542632
hp                -0.121996             -0.053848
qsec              -2.038089              0.077857


**4. Comparar resultados**

In [14]:
intervalo_normal = modelo.conf_int()
intervalo_normal.columns = ['Limite_I normal', 'Limite_S normal']


intervalo_boot_df = pd.DataFrame({
    'Limite_I Boot': Limite_I_boot,
    'Limite_S Boot': Limite_S_boot
}, index=['const', 'hp', 'qsec'])

tabla_comparativa = pd.concat([intervalo_normal, intervalo_boot_df], axis=1)

print(tabla_comparativa)

       Limite_I normal  Limite_S normal  Limite_I Boot  Limite_S Boot
const        25.614894        71.032516      28.196409      72.542632
hp           -0.113089        -0.056097      -0.121996      -0.053848
qsec         -1.979929         0.206770      -2.038089       0.077857


# **Aggregating**

Se usa aggregating porque un solo modelo es inestable, esto porque tenemos muchos factores y puede haber sobreajuste.
-> usamos muchos modelos y promediamos

**Actividad**

1. Train test 50% 50%
2. 1000 modelos con train
3. Escoger 3 variables (columnas) al azar
4. predict test, ytest, r2 test



Train

inicio -> vars=(3 al azar) -> Xi=X[vars] -> modelo;train -> end


Test

inicio -> xtest=xtest[vars] -> modelo;predict(Xtest) -> yi (predicción) -> end

In [15]:
df = df.drop(columns=['model'])

In [16]:
df.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [18]:
train, test = train_test_split(df, test_size=0.5, random_state=42)

In [19]:
variables = list(df.columns)
variables.remove('mpg')

In [20]:
predicciones = []

In [21]:
for i in range(1000):

    vars_random = np.random.choice(variables, size=3, replace=False)

    X_train = train[vars_random]
    y_train = train['mpg']

    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    X_test = test[vars_random]

    y_pred = modelo.predict(X_test)

    predicciones.append(y_pred)

pred_matrix = np.array(predicciones)

pred_matrix = pred_matrix.T

y_pred_prom = pred_matrix.mean(axis=1)


y_test = test['mpg'].values

r2_final = r2_score(y_test, y_pred_prom)

print("R2 final (promedio de modelos):", r2_final)

R2 final (promedio de modelos): 0.785616286108303


# **Random Forest Regressor**

Se toma una muestra -> K folds k=10 -> Random Forest Regressor -> R^2

In [34]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
!pip install scikit-optimize
from skopt import BayesSearchCV

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.8 MB/s eta 0:00:00


In [35]:
X = df.drop(columns=['mpg'])
y = df['mpg']

In [36]:
rf = RandomForestRegressor(random_state=42)

In [59]:
parametros = {
    'n_estimators': (5, 30),
    'max_depth': (5, 20),
    'max_leaf_nodes': (2, 20)
}

In [61]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [62]:
opt = BayesSearchCV(
    estimator=rf,
    search_spaces=parametros,
    n_iter=40,
    cv=kf,
    scoring='r2',
    random_state=42
)

In [63]:
opt.fit(X, y)

BayesSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
              estimator=RandomForestRegressor(random_state=42), n_iter=40,
              random_state=42, scoring='r2',
              search_spaces={'max_depth': (5, 20), 'max_leaf_nodes': (2, 20),
                             'n_estimators': (5, 30)})

In [64]:
print("Mejores parámetros:", opt.best_params_)
print("Mejor R2:", opt.best_score_)

Mejores parámetros: OrderedDict({'max_depth': 19, 'max_leaf_nodes': 4, 'n_estimators': 21})
Mejor R2: 0.46045224183259076


Con esos hiperparamteros que encontre agarro un nuevo forest y entreno el modelo con todos los datos y obtengo r2

In [65]:
rf2 = RandomForestRegressor(max_depth=2, max_leaf_nodes=17, n_estimators=24)

In [66]:
rf2.fit(X, y)

RandomForestRegressor(max_depth=2, max_leaf_nodes=17, n_estimators=24)

In [67]:
y_pred = rf2.predict(X)
y_pred

array([21.18222035, 21.18222035, 23.30190197, 20.36243433, 17.10486837,
       19.68555673, 14.87583732, 22.90681285, 21.77326532, 18.8982266 ,
       18.8982266 , 16.72957279, 16.72957279, 16.72957279, 13.72103822,
       13.72103822, 14.16687155, 30.05799603, 30.81143849, 31.72116071,
       21.77326532, 17.10486837, 17.10486837, 14.87583732, 17.10486837,
       28.70644511, 24.76372099, 27.68991147, 15.66845566, 20.49801065,
       14.87583732, 21.77326532])

In [70]:
r2 = r2_score(y,y_pred)
r2

0.9363300026678292

GradientBoostingRegressor Classifier

In [71]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from sklearn.ensemble import GradientBoostingRegressor


In [72]:
X = df.drop(columns=['mpg'])
y = df['mpg']

In [73]:
gb = GradientBoostingRegressor(random_state=42)

In [74]:
parametros = {
    'n_estimators': (5, 30),
    'max_depth': (5, 20),
    'max_leaf_nodes': (2, 20)
}

In [75]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [79]:
opt = BayesSearchCV(
    estimator=gb,
    search_spaces=parametros,
    n_iter=40,
    cv=kf,
    scoring='r2',
    random_state=42
)

In [80]:
opt.fit(X, y)

/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(20), np.int64(5), np.int64(22)] before, using random point [np.int64(9), np.int64(9), np.int64(6)]
  warnings.warn(


BayesSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
              estimator=GradientBoostingRegressor(random_state=42), n_iter=40,
              random_state=42, scoring='r2',
              search_spaces={'max_depth': (5, 20), 'max_leaf_nodes': (2, 20),
                             'n_estimators': (5, 30)})

In [81]:
print("Mejores parámetros:", opt.best_params_)
print("Mejor R2:", opt.best_score_)

Mejores parámetros: OrderedDict({'max_depth': 5, 'max_leaf_nodes': 5, 'n_estimators': 22})
Mejor R2: 0.48453101815852817


In [82]:
rf3 = RandomForestRegressor(max_depth=2, max_leaf_nodes=17, n_estimators=24)

In [83]:
rf3.fit(X, y)

RandomForestRegressor(max_depth=2, max_leaf_nodes=17, n_estimators=24)

In [84]:
y_pred3 = rf3.predict(X)
y_pred3

array([21.12664376, 21.12664376, 22.57991246, 19.3715715 , 16.64877492,
       18.63110433, 14.70871768, 22.37408456, 21.53158544, 18.61996339,
       18.61996339, 15.49163898, 15.49163898, 15.49163898, 13.78175339,
       13.53008672, 14.48758672, 29.73490728, 30.52847222, 30.9825    ,
       21.74900968, 16.36560392, 16.64877492, 14.70871768, 16.36560392,
       28.76863248, 25.00830289, 27.74405002, 16.2222864 , 20.80996747,
       14.70871768, 21.53158544])

In [85]:
r3 = r2_score(y,y_pred3)
r3

0.9278011544858872